# Notebook 1: Acréscimo de variáveis com Teoria dos Grafos

**Objetivo:** Construir uma rede de similaridade financeira e incorporar à base de dados. Utilizando o algoritmo K-Nearest Neighbors (KNN), vamos conectar clientes com comportamentos financeiros similares e extrair métricas topológicas (Grau, PageRank, Clustering e Comunidades de Louvain) para fornecer ao nosso modelo preditivo uma compreensão dos agrupamentos financeiros do banco.

# Importação de bibliotecas

In [1]:
# ==============================================================================
# 1. IMPORTAÇÃO DE BIBLIOTECAS E CONFIGURAÇÕES GLOBAIS
# ==============================================================================
# 1. Bibliotecas de manipulação de dados
import pandas as pd
import numpy as np
import os

# 2. Bibliotecas de Teoria dos Grafos e Machine Learning
import networkx as nx
import networkx.algorithms.community as nx_comm
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from scipy.stats import mode

# 3. Configurações visuais e globais
import warnings
warnings.filterwarnings("ignore")

# Carregamento da base de dados

In [2]:
# ==============================================================================
# 2. CARREGAMENTO DOS DADOS (LOAD)
# ==============================================================================
print(">>> Iniciando o carregamento da base original de clientes...")

file_path = '../data/raw/UCI_Credit_Card.csv'
df = pd.read_csv(file_path)

print(f"✅ Dados originais carregados com sucesso! Total: {df.shape[0]} clientes.\n")

target_col = 'default.payment.next.month'

df_train, df_test = train_test_split(
    df,
    test_size=0.3,
    random_state=42,
    stratify=df[target_col]
)

df_train = df_train.copy()
df_test = df_test.copy()

>>> Iniciando o carregamento da base original de clientes...
✅ Dados originais carregados com sucesso! Total: 30000 clientes.



# Seleção de features e padronização espacial

O algoritmo KNN calcula a distância Euclidiana entre os clientes. Se não padronizarmos os dados, variáveis com números absolutos muito grandes (como o limite de crédito ou o valor da fatura) ofuscarão variáveis menores e cruciais (como os meses de atraso). O `StandardScaler` nivela todas as métricas para a mesma régua matemática.

In [3]:
# ==============================================================================
# 3. SELEÇÃO DE FEATURES PARA A REDE E PADRONIZAÇÃO (TRANSFORM)
# ==============================================================================
print(">>> Isolando features financeiras e aplicando padronização (Z-score)...")

# 1. Isolamento das variáveis estritamente financeiras/comportamentais
financial_features = [
    'LIMIT_BAL', 
    'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6',
    'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6',
    'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6'
]

X_fin_train = df_train[financial_features]
X_fin_test = df_test[financial_features]

# 2. Padronização espacial
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_fin_train)
X_test_scaled = scaler.transform(X_fin_test)

print("✅ Features financeiras padronizadas com sucesso!\n")

>>> Isolando features financeiras e aplicando padronização (Z-score)...
✅ Features financeiras padronizadas com sucesso!



# Construção do grafo de similaridade

Nesta etapa, instruímos o algoritmo a mapear a posição de cada cliente no espaço multidimensional e conectar-se aos seus 5 vizinhos mais próximos. O resultado é transformado em um grafo não direcionado, revelando as relações ocultas de comportamentos de consumo.

In [4]:
# ==============================================================================
# 4. CONSTRUÇÃO DO GRAFO DE SIMILARIDADE (KNN)
# ==============================================================================
print(">>> Calculando os vizinhos mais próximos para estruturação da rede...")

# 1. Busca espacial: n_neighbors=6 (o cliente + os seus 5 vizinhos reais)
knn = NearestNeighbors(n_neighbors=6, metric='euclidean', n_jobs=-1)
knn.fit(X_train_scaled)

# 2. Extração da matriz de adjacência (conexões matemáticas)
adj_matrix = knn.kneighbors_graph(X_train_scaled, mode='connectivity')

# 3. Geração do objeto Grafo no NetworkX
G = nx.from_scipy_sparse_array(adj_matrix)

# 4. Limpeza: Remoção de auto-conexões (o cliente ligado a si mesmo)
G.remove_edges_from(nx.selfloop_edges(G))

print(f"✅ Grafo construído com sucesso!")
print(f"📊 Estrutura da Rede -> Nós (Clientes): {G.number_of_nodes()} | Arestas (Similaridades): {G.number_of_edges()}\n")

>>> Calculando os vizinhos mais próximos para estruturação da rede...
✅ Grafo construído com sucesso!
📊 Estrutura da Rede -> Nós (Clientes): 21000 | Arestas (Similaridades): 79272



# Extração de métricas topológicas

Transformamos as conexões geométricas em atributos numéricos interpretáveis por algoritmos de aprendizado de máquina. Calculamos a centralidade (PageRank), a popularidade do comportamento (Degree), a formação de bolhas (Clustering) e as comunidades de comportamento (Comunidades de Louvain).

In [5]:
# ==============================================================================
# 5. EXTRAÇÃO DE MÉTRICAS DE REDE (ENRIQUECIMENTO DE VARIÁVEIS)
# ==============================================================================
print(">>> Extraindo métricas topológicas da rede (isso pode levar alguns instantes)...")

# 1. PageRank (centralidade e importância do comportamento)
pagerank = nx.pagerank(G, alpha=0.85)
pagerank_dict = dict(zip(df_train.index, pagerank.values()))
df_train['network_pagerank'] = df_train.index.map(pagerank_dict)
print(" • PageRank calculado.")

# 2. Grau / Degree (quão comum é esse perfil na base inteira)
degree_dict_raw = dict(G.degree())
degree_dict = dict(zip(df_train.index, degree_dict_raw.values()))
df_train['network_degree'] = df_train.index.map(degree_dict)
print(" • Grau de conectividade calculado.")

# 3. Clustering Coefficient (tendência de formação de grupos financeiros)
clustering = nx.clustering(G)
clustering_dict = dict(zip(df_train.index, clustering.values()))
df_train['network_clustering'] = df_train.index.map(clustering_dict)
print(" • Coeficiente de agrupamento calculado.")

# 4. Detecção de Comunidades via Louvain (comunidades financeiras ocultas)
print(">>> Detectando comunidades de comportamento financeiro (Louvain)...")
communities = nx_comm.louvain_communities(G, seed=42)

# Mapeando qual nó pertence a qual comunidade para integrar ao DataFrame
community_mapping = {}
for comm_id, comm_nodes in enumerate(communities):
    for node in comm_nodes:
        community_mapping[node] = comm_id

community_mapping_df = dict(zip(df_train.index, community_mapping.values()))
df_train['network_louvain'] = df_train.index.map(community_mapping_df)

print(f"✅ Extração concluída! Foram identificadas {len(communities)} comunidades de comportamento financeiro distintas.\n")

>>> Extraindo métricas topológicas da rede (isso pode levar alguns instantes)...
 • PageRank calculado.
 • Grau de conectividade calculado.
 • Coeficiente de agrupamento calculado.
>>> Detectando comunidades de comportamento financeiro (Louvain)...
✅ Extração concluída! Foram identificadas 66 comunidades de comportamento financeiro distintas.



In [6]:
# ==============================================================================
# 5.1 PROJEÇÃO DAS MÉTRICAS DE REDE NO CONJUNTO DE TESTE
# ==============================================================================
print(">>> Projetando métricas de rede para o conjunto de teste...")

# ----------------------------------------------------------------------
# CONTEXTO:
# O grafo foi construído apenas com os dados de treino.
# Portanto, os clientes do conjunto de teste não possuem métricas de rede
# (PageRank, Degree, Clustering, Louvain), pois não estão inseridos no grafo.
#
# SOLUÇÃO:
# Para contornar isso, utilizamos o KNN treinado no espaço original
# para encontrar, para cada cliente de teste, os k vizinhos mais próximos
# dentro do conjunto de treino.
#
# A ideia é assumir que clientes com comportamento financeiro semelhante
# (no espaço de features) também terão propriedades de rede semelhantes.
#
# Assim, projetamos as métricas de rede do treino para o teste usando:
# - Média (para variáveis contínuas)
# - Moda (para variável categórica - Louvain)
# ----------------------------------------------------------------------

# Para cada cliente de teste, retorna os índices dos vizinhos no treino
distances, indices = knn.kneighbors(X_test_scaled)

# Conversão das métricas de rede para listas indexáveis
pagerank_values = list(pagerank.values())
degree_values = list(degree_dict_raw.values())
clustering_values = list(clustering.values())
community_values = list(community_mapping.values())

# 1. Projeção das features contínuas (média dos vizinhos)

# PageRank médio dos vizinhos
df_test['network_pagerank'] = [
    np.mean([pagerank_values[i] for i in neigh]) for neigh in indices
]

# Grau médio dos vizinhos
df_test['network_degree'] = [
    np.mean([degree_values[i] for i in neigh]) for neigh in indices
]

# Clustering médio dos vizinhos
df_test['network_clustering'] = [
    np.mean([clustering_values[i] for i in neigh]) for neigh in indices
]

# 2. Projeção da feature categórica (comunidade Louvain)

# Atribuímos a comunidade mais frequente entre os vizinhos (moda)
df_test['network_louvain'] = [
    mode([community_values[i] for i in neigh], keepdims=True)[0][0]
    for neigh in indices
]

print("✅ Projeção concluída!\n")

>>> Projetando métricas de rede para o conjunto de teste...
✅ Projeção concluída!



# Exportação dos dados enriquecidos

Com as variáveis de rede devidamente anexadas ao histórico de cada cliente, exportamos o novo dataset em formato `.parquet`.

In [7]:
# ==============================================================================
# 6. EXPORTAÇÃO DOS DADOS ENRIQUECIDOS (LOAD)
# ==============================================================================
print(">>> Exportando a nova base de dados...")

# 1. Garantia de existência do diretório
os.makedirs('../data/processed', exist_ok=True)
output_path = '../data/processed/credit_data_with_network.parquet'

# 2. Exportação em formato .parquet
df_final = pd.concat([df_train, df_test]).sort_index()
df_final.to_parquet(output_path, index=False)

print(f"✅ Arquivo exportado com sucesso para: {output_path}\n")

print(">>> Amostra do novo dataset com as métricas de rede:")

target_name = 'default.payment.next.month'
display(df_final[['LIMIT_BAL', target_name, 'network_pagerank', 'network_degree', 'network_clustering', 'network_louvain']].head())

>>> Exportando a nova base de dados...
✅ Arquivo exportado com sucesso para: ../data/processed/credit_data_with_network.parquet

>>> Amostra do novo dataset com as métricas de rede:


,LIMIT_BAL,default.payment.next.month,network_pagerank,network_degree,network_clustering,network_louvain
0,20000.0,1,0.000036,5.000000,0.300000,39
1,120000.0,1,0.000041,5.333333,0.377778,3
2,90000.0,0,0.000040,6.000000,0.066667,64
3,50000.0,0,0.000065,12.333333,0.405609,28
4,50000.0,0,0.000033,5.000000,0.200000,4
